In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# If True, test is running on Colab. Otherwise, test if assumed to be offline.
TEST_ON_COLAB = False
FOLDERNAME = None  # only used if TEST_ON_COLAB is True

assert not (FOLDERNAME is None and TEST_ON_COLAB), "FOLDERNAME has to be set if TEST_ON_COLAB is True"

# Atividade de programação 2 (AP2)
## Redes neurais
### Valor: 15 pontos

Responda as questões indicadas e preencha os campos indicados com o tag `TODO`.

Para essa atividade, você deverá utilizar o mesmo dataset de classificação utilizado no AP1. Caso não tenha realizado essa atividade, pesquise um dataset de imagens de sua preferência (exceto mnist) utilizando o site `https://openml.org/`. O dataset será baixado através de função `fetch_openml` da biblioteca `scikit-learn`. O dataset deve apresentar a mesma estrutura de pixels/labels ($X$, $y$) vistos em exemplos de aula. Caso não esteja assim, faça as devidas transformações.

In [ ]:
# TODO: Set the dataset ID #################
DATASET_OPENML_ID = None
############################################

assert DATASET_OPENML_ID is not None, "DATASET_OPENML_ID is not set"

Utilizaremos uma semente para garantir a reprodutibilidade dos resultados.

In [ ]:
SEED = 42

The code below loads the dataset.

In [ ]:
# Create dataset directory
import os

if TEST_ON_COLAB:
    # This mounts your Google Drive to the Colab VM.
    from google.colab import drive

    drive.mount("/content/drive")
    cache_dir = f"/content/drive/My Drive/{FOLDERNAME}/dataset/{DATASET_OPENML_ID}"
else:
    cache_dir = f"dataset/{DATASET_OPENML_ID}"

os.makedirs(cache_dir, exist_ok=True)

# Fetching the dataset
from sklearn.datasets import fetch_openml
import numpy as np

X_file_path = f"{cache_dir}/X.npy"
y_file_path = f"{cache_dir}/y.npy"

# Check if the dataset files already exist
if not (os.path.exists(X_file_path) and os.path.exists(y_file_path)):
    # Fetch the dataset where X is the data and y is the target
    X, y = fetch_openml(DATASET_OPENML_ID, as_frame=False, cache=True, return_X_y=True)

    # Save the dataset as numpy arrays
    np.save(X_file_path, X.astype(np.float32))
    np.save(y_file_path, y)
    print(f"{DATASET_OPENML_ID} dataset downloaded and saved successfully to {cache_dir}.")
else:
    X = np.load(X_file_path, allow_pickle=True)
    y = np.load(y_file_path, allow_pickle=True)
    print(f"{DATASET_OPENML_ID} dataset already exists in {cache_dir}. Skipping download.")

print(X.shape)
print(X.min(), X.max())

Visualizando alguns amostras do dataset:

In [ ]:
import matplotlib.pyplot as plt

# Visualize some examples from the dataset.
# We show a few examples of training images from each class.
classes = [int(class_id) for class_id in np.unique(y)]
num_classes = len(classes)
samples_per_class = 7
for cls in classes:
    idxs = np.flatnonzero(y == str(cls))
    idxs = np.random.choice(idxs, samples_per_class, replace=False)
    for i, idx in enumerate(idxs):
        plt_idx = i * num_classes + cls + 1
        plt.subplot(samples_per_class, num_classes, plt_idx)
        plt.imshow(X[idx].reshape((28, -1)).astype('uint8'), cmap='gray')
        plt.axis('off')
        if i == 0:
            plt.title(cls)
plt.show()

**1. (1 ponto)** Agora, você precisa particionar o conjunto de dados em treino e teste (a criação de um conjunto de validação será realizada implicitamente durante o treinamento com a biblioteca `scikit-learn`).

Utilize a função `train_test_split` do módulo `sklearn.model_selection` para dividir o conjunto de dados (`X` e `y`): 90% dos dados para treinamento e 10% para teste.

In [ ]:
from sklearn.model_selection import train_test_split

# Convert labels from string to integer
y = y.astype(int)

# Set the random seed for reproducibility
np.random.seed(SEED)

# TODO: Split the dataset into train and and test sets #################
X_train, X_test, y_train, y_test = None, None, None, None
########################################################################

assert X_train is not None, "X_train is not set"
assert y_train is not None, "y_train is not set"
assert X_test is not None, "X_test is not set"
assert y_test is not None, "y_test is not set"

print(f"Train set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

## Treinamento e avaliação de um modelo de rede neural utilizando scikit-learn

O código a seguir mostra com instanciar e treinar uma rede neural com uma camada oculta utilizandom a biblioteca `scikit-learn`. A rede possui as seguintes configurações: uma camada oculda de 128 neurônios, função de ativação ReLU e taxa de aprendizado inicial 0.001. Perceba, no exemplo em questão, que o treinamento encerra antes do critério de convergência determinado pelo mecanismo de **early stopping** (leia a documentação da ckasse`MLPClassifier` para entender como o mecanismo funciona).

In [ ]:
from sklearn.neural_network import MLPClassifier

# Define the MLPClassifier with early stopping
mlp = MLPClassifier(hidden_layer_sizes=(128,), activation='relu', max_iter=30, learning_rate_init=0.001, verbose=True, random_state=SEED)

# Train the model
mlp.fit(X_train, y_train)

print(f"Best iteration: {mlp.n_iter_}")

A seguir, temos o resultado da avaliação do modelo treinado.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

# Predict the test set
y_pred = mlp.predict(X_test)

# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=[class_id for class_id in range(10)])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[class_id for class_id in range(10)])
disp.plot()
plt.show()

# Compute classification report
class_report = classification_report(y_test, y_pred, target_names=[str(class_id) for class_id in range(10)])

print("\nClassification Report:")
print(class_report)

## Avaliando a sensibilidade de parâmetros

Para analisar a sensibilidade do modelo em relação a um determinando parâmetro, você pode treinar e avaliar o modelo variando o valor desse parâmetro fixando os demais (abordagem conhecida como One-at-a-time (OAT)). Nas questões a seguir, você deve variar o número de neurônios na camada oculta e a taxa de aprendizado.

**2. (3 pontos)** Analise a sensibilidade do modelo em relação ao número de neurônios na camada oculta. Varie o número de neurônios em potências de 2 (2, 4, 8, 16, ..., 1024, 2048). Para cada valor de neurônios, treine um modelo e avalie-o no conjunto de teste. Armazene a acurácia de cada modelo em uma lista. A acurácia do modelo treinado pode ser calculada utilizando a função `accuracy_score` do módulo `sklearn.metrics`.

In [ ]:
from sklearn.metrics import accuracy_score

accuracy_history = []
num_hidden_units_history = [2 **d  for d in range(1, 12)]

# TODO: Train the MLPClassifier with different number of hidden units and store the accuracy #############################################################
#
# COMPLETE THE CODE
#
##########################################################################################################################################################

assert len(accuracy_history) == len(num_hidden_units_history), "accuracy_history and num_hidden_units_history should have the same length"    

In [ ]:
# Plot the accuracy vs. number of hidden units
plt.plot(num_hidden_units_history, accuracy_history)
plt.xlabel('Number of Hidden Units')
plt.ylabel('Accuracy')
plt.title('Accuracy vs. Number of Hidden Units')
plt.show()

**3. (3 pontos)** Agora, analise a sensibilidade do modelo em relação à taxa de aprendizado inicial (`learning_rate_init`) considerando os seguintes valores: 0.0001, 0.001, 0.01, 0.1. Para cada valor, treine um modelo e avalie-o no conjunto de teste. Armazene a acurácia de cada modelo em uma lista.

In [ ]:
from sklearn.metrics import accuracy_score

accuracy_history = []
learning_rate_init_history = [10 ** (-d) for d in range(4, 0, -1)]

# TODO: Train the MLPClassifier with different initial learning rates and record the accuracy #############################################################
for learning_rate_init in learning_rate_init_history:
#
# COMPLETE THE CODE
#
###########################################################################################################################################################    

assert len(learning_rate_init_history) == len(accuracy_history), "The history of learning rates and accuracies should have the same length."

In [ ]:
# Plot the accuracy vs. learning rate
plt.plot(learning_rate_init_history, accuracy_history)
plt.xlabel('Learning Rate')
plt.ylabel('Accuracy')
plt.title('Accuracy vs. Learning Rate')
plt.xscale('log')
plt.show()

## Escolhendo o melhor modelo no conjunto de validação

Nessa parte do trabalho, o objetivo é treinar o modelo utilizando uma partição explícita de validação. Ao final de cada época, o modelo dever ser avaliado na partição de validação. O modelo da época que resultar na maior acurácia será utilizado para avaliação no conjunto de teste.

**4. (2 pontos)** Particione o conjunto de dados em treino, validação e teste. 

Use a função `train_test_split` do módulo `sklearn.model_selection` para dividir o conjunto de dados (`X` e `y`) em um conjunto de treinamento e um conjunto temporário, com 80% dos dados para treinamento e 20% para o conjunto temporário.

Em seguida, divida o conjunto temporário igualmente em conjuntos de validação e teste. Portanto, no final, você deve ter 80% dos dados para treinamento, 10% para validação e 10% para teste.


In [ ]:
from sklearn.model_selection import train_test_split

# Set the random seed for reproducibility
np.random.seed(SEED)

# Convert labels from string to integer

# TODO: Split the dataset into train, validation, and test sets ##############
X_train, X_temp, y_train, y_temp = None, None, None, None
X_val, X_test, y_val, y_test = None, None, None, None
##############################################################################

assert X_train is not None, "X_train is not set"
assert y_train is not None, "y_train is not set"
assert X_val is not None, "X_val is not set"
assert y_val is not None, "y_val is not set"
assert X_test is not None, "X_test is not set"
assert y_test is not None, "y_test is not set"

print(f"Train set size: {len(X_train)}")
print(f"Val set size: {len(X_val)}")
print(f"Test set size: {len(X_test)}")

 **5. (6 pontos)** Com os dados em mão, é hora de treinar e testar o modelo.

O treinamento na forma proposta demanda um loop que treina o modelo por *uma interação* e avalia o modelo na partição de validação. O modelo é salvo em arquivo (`best-model.joblib`) se a acurácia na partição de validação for maior que a melhor acurácia obtida até o momento. O treinamento é interrompido após um número máximo de épocas (100). Ao final, o modelo deve ser carrergado do arquivo e avaliado no conjunto de teste.

Para o treinamento, deve ser habilitar a opção `warm_start` na instanciação do modelo, que permite treinar o modelo por uma interação e continuar o treinamento a partir do ponto em que parou. A função `partial_fit` do modelo deve ser utilizada para treinar o modelo por uma interação.

Os parâmetros do modelo devem ser os seguintes:
- Número de neurônios na camada oculta: 2048
- Taxa de aprendizado inicial: 0.001
- Função de ativação: 
- Número máximo de épocas: 30


In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import joblib # This helps to save the model

# TODO: Set the parameters #################
num_epochs = None
num_hidden_units = None
learning_rate_init = None
activation = None
############################################

# Define the MLPClassifier (notice that the number of iterations was not defined)
mlp = MLPClassifier(
    hidden_layer_sizes=(num_hidden_units,),
    activation=activation,
    learning_rate_init=learning_rate_init,
    warm_start=True,
    verbose=True,
    random_state=SEED,
    early_stopping=False,
    n_iter_no_change=1000, # large number of iterations to avoid early stopping message
)

# Best configuration
best_accuracy = 0

# TODO: Train the model for num_epochs #######################################
#
# COMPLETE THE CODE
#
##############################################################################

assert best_accuracy > 0, "best_accuracy is not set"
assert os.path.exists("best-model.joblib"), "The best model was not saved"

# Load the best model
best_model = joblib.load("best-model.joblib")

# Evaluate the best model
accuracy = best_model.score(X_test, y_test)

# Predictions
y_pred = best_model.predict(X_test)

# Compute confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred, labels=[class_id for class_id in range(10)])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[class_id for class_id in range(10)])
disp.plot()
plt.show()

# Compute classification report
class_report = classification_report(y_test, y_pred, target_names=[str(class_id) for class_id in range(10)])

print("\nClassification Report:")
print(class_report)